# Documentação — Notebook 00_setup_config

Este notebook é o **ponto de partida de todo o pipeline**. Ele carrega as credenciais do arquivo `.env` para dentro das variáveis de ambiente do Python, para que todos os outros notebooks consigam acessar o SQL Server e o ADLS sem precisar digitar senhas.

Todos os notebooks do projeto começam com `%run ./00_setup_config` para herdar as credenciais carregadas aqui.

## Célula 1 — Importação de bibliotecas

```python
from dotenv import load_dotenv
import os
```

**O que faz:** Importa duas bibliotecas essenciais:
- `load_dotenv` (da biblioteca `python-dotenv`) — lê arquivos `.env` e transforma cada linha em uma variável de ambiente
- `os` — biblioteca padrão do Python para acessar variáveis de ambiente com `os.getenv("NOME_DA_VARIAVEL")`

**Por que é importante:** Sem essas importações, nenhum dos outros notebooks consegue acessar as credenciais. É a base de tudo.

## Célula 2 — Identificação do diretório atual

```python
from pathlib import Path
current_path = Path.cwd()
print("Diretório atual:")
print(current_path)
```

**O que faz:** Mostra em qual pasta o notebook está executando usando `Path.cwd()` (current working directory).

**Por que é importante:** Ajuda a confirmar que o notebook está rodando no diretório correto do projeto. Em ambientes serverless, o diretório pode variar, então é útil para debug.

## Célula 3 — Busca dinâmica do arquivo .env

```python
workspace_path = Path("/Workspace")
env_files = list(workspace_path.rglob(".env"))
print("Arquivos .env encontrados:")
for env_file in env_files:
    print(env_file)
```

**O que faz:** Procura recursivamente (em todas as subpastas do `/Workspace`) por arquivos chamados `.env` usando `rglob()`.

**Por que é importante:** A busca é dinâmica — não é preciso saber o caminho exato do `.env`. Isso torna o notebook portável: funciona independente de onde o projeto está clonado.

**Resultado esperado:** Lista o caminho do `.env` do projeto, algo como:
`/Workspace/Repos/engenheiro_leandroreis@hotmail.com/estagio-empregadados-turma-2/.env`

## Célula 4 — Carregamento do .env

```python
env_path = next(Path("/Workspace").rglob(".env"), None)
if env_path:
    load_dotenv(env_path)
    print(f".env carregado de: {env_path}")
else:
    print(".env não encontrado.")
```

**O que faz:** Este é o **momento chave** do notebook. Usa `next()` para pegar o primeiro `.env` encontrado na busca e chama `load_dotenv()` para carregá-lo.

**O que acontece por baixo dos panos:**
1. `load_dotenv()` lê cada linha do arquivo `.env`
2. Linhas no formato `CHAVE=valor` são transformadas em variáveis de ambiente
3. A partir daqui, `os.getenv("SQL_HOST")`, `os.getenv("SQL_PASSWORD")`, etc. funcionam em qualquer célula

**Resultado esperado:** `.env carregado de: /Workspace/Repos/.../.env`

## Célula 5 — Confirmação das credenciais

```python
variaveis = [
    "ADLS_CLIENT_ID",
    "ADLS_TENANT_ID",
    "ADLS_CLIENT_SECRET",
    "SQL_HOST",
    "SQL_DATABASE",
    "SQL_USERNAME",
    "SQL_PASSWORD"
]
for variavel in variaveis:
    valor = os.getenv(variavel)
    print(f"{variavel}: {'OK' if valor else 'NÃO ENCONTRADA'}")
```

**O que faz:** Verifica cada uma das 7 credenciais esperadas, uma por uma.

**Boa prática de segurança:** Mostra apenas `OK` ou `NÃO ENCONTRADA` — **nunca imprime o valor da senha**. Isso evita que credenciais fiquem expostas no output do notebook.

**Resultado esperado:**
```
ADLS_CLIENT_ID: OK
ADLS_TENANT_ID: OK
ADLS_CLIENT_SECRET: OK
SQL_HOST: OK
SQL_DATABASE: OK
SQL_USERNAME: OK
SQL_PASSWORD: OK
```

Se alguma variável mostrar `NÃO ENCONTRADA`, verifique se o arquivo `.env` está na pasta do projeto e se contém todas as chaves.

## Resumo do fluxo

| Célula | Função | Importância |
| --- | --- | --- |
| 1 | Importa `load_dotenv` e `os` | Base — sem essas libs nada funciona |
| 2 | Mostra diretório atual | Debug — confirma onde está rodando |
| 3 | Busca arquivos `.env` no Workspace | Busca dinâmica — não precisa saber o caminho |
| 4 | Carrega o `.env` com `load_dotenv()` | **Momento chave** — popula as variáveis de ambiente |
| 5 | Confirma se cada credencial foi carregada | Validação — mostra OK sem expor senhas |

**Fluxo:** Importa → busca → carrega → confirma

Após a execução, qualquer notebook que fizer `%run ./00_setup_config` terá todas as credenciais disponíveis em `os.getenv()`.

## Variáveis do .env

| Variável | Uso |
| --- | --- |
| `ADLS_CLIENT_ID` | Client ID da Service Principal do Azure (ADLS Gen2) |
| `ADLS_TENANT_ID` | Tenant ID do Azure (ADLS Gen2) |
| `ADLS_CLIENT_SECRET` | Secret da Service Principal do Azure (ADLS Gen2) |
| `SQL_HOST` | Host do Azure SQL Server |
| `SQL_DATABASE` | Nome do banco de dados (internshipDatabase) |
| `SQL_USERNAME` | Usuário do SQL Server |
| `SQL_PASSWORD` | Senha do SQL Server |

**Nota de segurança:** O arquivo `.env` está listado no `.gitignore` do projeto e não deve ser commitado para o GitHub.